In [1]:
from faker import Faker
fake = Faker('en_GB')
Faker.seed(42)

In [2]:
n_sensors = 50
n_readings = 5000

In [3]:
from unidecode import unidecode
import uuid

In [4]:
from datetime import datetime, timedelta

In [5]:
from faker.providers import DynamicProvider

sensor = {}
sensor_type = ['temperature', 'humidity', 'air_quality']

unique_districts = set()
while len(unique_districts) < 15:
    unique_districts.add(fake.administrative_unit())

# 2. Register these 15 districts into a DynamicProvider to override the default method
district_provider = DynamicProvider(
     provider_name="administrative_unit",
     elements=list(unique_districts),
)
fake.add_provider(district_provider)

for i in range(n_sensors):
    latitude = fake.latitude()
    longitude = fake.longitude()
    sensor[i] = {
        'sensor_id': str(uuid.uuid4()),
        'district': fake.administrative_unit(),
        'sensor_type': fake.random_element(elements=sensor_type),
        'latitude_longitude': f"{latitude}, {longitude}",
        'installed_at': fake.date_time()
    }

print(sensor[0])

{'sensor_id': 'cf86f7cd-9077-4414-88c5-82918e062b8c', 'district': 'Moray', 'sensor_type': 'temperature', 'latitude_longitude': '-27.5455665, 91.310554', 'installed_at': datetime.datetime(2001, 9, 25, 0, 29, 32, 984469)}


In [6]:
sensor_id = [sensor[i]['sensor_id'] for i in range(n_sensors)]

In [7]:
readings = {}

for i in range(n_readings):
    readings[i] = {
        'reading_id': str(uuid.uuid4()),
        'sensor_id': sensor_id[i % n_sensors],
        'value': round(fake.random.gauss(mu=20, sigma=5), 2) if sensor[i % n_sensors]['sensor_type'] == 'temperature' else (round(fake.random.gauss(mu=50, sigma=10), 2) if sensor[i % n_sensors]['sensor_type'] == 'humidity' else round(fake.random.gauss(mu=100, sigma=20), 2)),
        'unit': '°C' if sensor[i % n_sensors]['sensor_type'] == 'temperature' else ('%' if sensor[i % n_sensors]['sensor_type'] == 'humidity' else 'AQI'),
        'recorded_at': fake.date_time_between(start_date=sensor[i % n_sensors]['installed_at'], end_date=datetime.now()),
        'is_anomaly': fake.boolean(chance_of_getting_true=7)
    }

print(readings[0])

{'reading_id': 'e79573df-f114-417c-9329-1251e99e5a6f', 'sensor_id': 'cf86f7cd-9077-4414-88c5-82918e062b8c', 'value': 20.01, 'unit': '°C', 'recorded_at': datetime.datetime(2012, 9, 22, 0, 29, 49, 221138), 'is_anomaly': False}


In [8]:
import pandas as pd

In [9]:
sensors_df = pd.DataFrame.from_dict(sensor, orient='index')
readings_df = pd.DataFrame.from_dict(readings, orient='index')

In [10]:
sensors_df.to_csv('sensors_raw.csv', index=False)
readings_df.to_csv('readings_raw.csv', index=False)